This script is a high-performance geospatial processing pipeline designed to transform millions of overlapping machine readings into high-fidelity Cloud Optimized GeoTIFFs (COG).

It specifically addresses the challenges of "machine-pass" data—where high-density, overlapping 0.25m x 0.25m polygons must be flattened into a meaningful grid without losing data integrity or suffering from spatial drift.

## Core Features
Multi-Mode Analytics: Toggle between three distinct calculation modes:

max: Captures the peak temperature recorded in each cell.

min: Captures the lowest temperature recorded in each cell.

passes: Generates a "density map" counting the total number of times a machine footprint touched each cell.

High-Speed Ingestion: Utilizes DuckDB and the Spatial Extension to filter and project millions of rows from Parquet files in seconds.

Strict Grid Anchoring: Implements mathematical bounding-box snapping to ensure that every pixel is anchored to a global 0.25m grid, preventing "grid shift" when comparing different datasets.

Rust-Accelerated Parsing: Uses the orjson library to bypass standard Python string bottlenecks, processing geometry data at near-native speeds.

Memory Optimized: Employs generator expressions and rasterio.MemoryFile to keep the RAM footprint minimal, even when processing massive fields.

### How to Use
Simply adjust the Configuration block at the top of the script:

MACHINE_PREFIX: Filter for specific machine groups (e.g., "F_" or "M_").

MODE: Select your calculation logic ("max", "min", or "passes").

CELL_SIZE: Define your output resolution (defaulted to 0.25 meters).

In [28]:
import duckdb
import rasterio
import orjson
import math
import time
import os
from rasterio import features
from rasterio.enums import MergeAlg
from rasterio.transform import from_bounds
from rio_cogeo.cogeo import cog_translate
from rio_cogeo.profiles import cog_profiles

# ==========================================
# CONFIGURATION
# ==========================================
MACHINE_PREFIX = "M_"   # Choices: "F_" or "M_"
MODE = "passes"            # Choices: "max", "min", "passes"
CELL_SIZE = 0.25        # 0.25m resolution
INPUT_FILE = "./area_15.parquet"
# ==========================================

# --- PROFILER SETUP ---
class Profiler:
    def __init__(self):
        self.start_time = time.perf_counter()
        self.last_time = self.start_time

    def mark(self, message):
        now = time.perf_counter()
        print(f"[{now - self.start_time:06.3f}s TOTAL | {now - self.last_time:06.3f}s STEP] {message}")
        self.last_time = now

p = Profiler()
p.mark(f"Script started. Mode: {MODE} | Filter: {MACHINE_PREFIX}%")

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")

# 1. GET ANCHORED BOUNDS
# We query the bounds for the specific machine filter
minx, miny, maxx, maxy = con.execute(f"""
    SELECT 
        MIN(ST_XMin(geom)), MIN(ST_YMin(geom)), 
        MAX(ST_XMax(geom)), MAX(ST_YMax(geom))
    FROM (
        SELECT ST_Transform(geometry, 'OGC:CRS84', 'EPSG:32631') as geom 
        FROM read_parquet('{INPUT_FILE}')
        WHERE machine LIKE '{MACHINE_PREFIX}%'
    )
""").fetchone()

# Strict Grid Anchoring
minx = math.floor(minx / CELL_SIZE) * CELL_SIZE
miny = math.floor(miny / CELL_SIZE) * CELL_SIZE
maxx = math.ceil(maxx / CELL_SIZE) * CELL_SIZE
maxy = math.ceil(maxy / CELL_SIZE) * CELL_SIZE

width = math.ceil((maxx - minx) / CELL_SIZE)
height = math.ceil((maxy - miny) / CELL_SIZE)
transform = from_bounds(minx, miny, maxx, maxy, width, height)
p.mark(f"Grid calculated: {width}x{height} pixels.")

# 2. CONFIGURE MODE PARAMETERS
# Logic for Last-Value-Wins (Replace) vs Cumulative (Add)
if MODE == "max":
    val_col = "temperature"
    order_by = "ORDER BY temperature ASC"  # High values written last
    merge_alg = MergeAlg.replace
elif MODE == "min":
    val_col = "temperature"
    order_by = "ORDER BY temperature DESC" # Low values written last
    merge_alg = MergeAlg.replace
elif MODE == "passes":
    val_col = "1"
    order_by = ""                           # Order doesn't matter for counting
    merge_alg = MergeAlg.add
else:
    raise ValueError("Invalid MODE. Choose 'min', 'max', or 'passes'.")

# 3. EXTRACT DATA
query = f"""
    SELECT 
        ST_AsGeoJSON(ST_Transform(geometry, 'OGC:CRS84', 'EPSG:32631')) AS geom, 
        {val_col} AS val 
    FROM read_parquet('{INPUT_FILE}')
    WHERE machine LIKE '{MACHINE_PREFIX}%'
      AND temperature IS NOT NULL
      AND geometry IS NOT NULL
    {order_by}
"""
data = con.execute(query).fetchall() 
p.mark(f"DuckDB Extraction finished. Rows: {len(data)}")

# 4. RASTERIZE
shapes = ((orjson.loads(row[0]), row[1]) for row in data)

raster = features.rasterize(
    shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0, 
    all_touched=True,             
    merge_alg=merge_alg,   
    dtype='float32' 
)
p.mark(f"Rasterization ({MODE}) finished.")

# 5. COG PIPELINE
output_filename = f"machine_{MACHINE_PREFIX}_{MODE}.tif"

with rasterio.MemoryFile() as memfile:
    with memfile.open(
        driver="GTiff", height=height, width=width, count=1, 
        dtype='float32', crs="EPSG:32631", transform=transform, nodata=0 
    ) as mem_dataset:
        mem_dataset.write(raster, 1)
    p.mark("In-memory TIFF created.")
        
    with memfile.open() as src_dataset:
        dst_profile = cog_profiles.get("deflate")
        cog_translate(src_dataset, output_filename, dst_kwargs=dst_profile, quiet=True)

p.mark(f"Finished! COG saved as: {output_filename}")
con.close()

[00.000s TOTAL | 00.000s STEP] Script started. Mode: passes | Filter: M_%
[00.294s TOTAL | 00.294s STEP] Grid calculated: 16929x27392 pixels.
[02.170s TOTAL | 01.876s STEP] DuckDB Extraction finished. Rows: 606314
[10.674s TOTAL | 08.504s STEP] Rasterization (passes) finished.
[13.668s TOTAL | 02.993s STEP] In-memory TIFF created.
[24.433s TOTAL | 10.766s STEP] Finished! COG saved as: machine_M__passes.tif
